In [1]:

import sqlite3 
import pandas as pd 
  
con = sqlite3.connect('../data/raw/ipl.db') 
  
def q(sql): 
    """Run a SELECT and show the result as a table.""" 
    return pd.read_sql(sql, con)


def run_sql_file(path): 
    """Read a .sql file and execute every statement in it.""" 
    with open(path, 'r') as f: 
        script = f.read() 
    con.executescript(script) 
    con.commit() 
    print('ran', path)

In [2]:
q("SELECT * FROM matches LIMIT 5;")

,match_id,season_id,balls_per_over,city,match_date,event_name,match_number,gender,match_type,format,...,match_winner,win_by_runs,win_by_wickets,result,player_of_match,team1_id,team2_id,toss_winner_id,match_winner_id,player_of_match_id
0,335982,2008,6,Bangalore,2008-04-18,Indian Premier League,1.0,male,T20,T20,...,Kolkata Knight Riders,140.0,NaN,win,BB McCullum,1,6,1,6,46.0
1,1082591,2017,6,Hyderabad,2017-04-05,Indian Premier League,1.0,male,T20,T20,...,Sunrisers Hyderabad,35.0,NaN,win,Yuvraj Singh,2,1,1,2,15.0
2,1082592,2017,6,Pune,2017-04-06,Indian Premier League,2.0,male,T20,T20,...,Rising Pune Supergiant,NaN,7.0,win,SPD Smith,4,3,4,4,36.0
3,1082593,2017,6,Rajkot,2017-04-07,Indian Premier League,3.0,male,T20,T20,...,Kolkata Knight Riders,NaN,10.0,win,CA Lynn,5,6,6,6,57.0
4,1082594,2017,6,Indore,2017-04-08,Indian Premier League,4.0,male,T20,T20,...,Punjab Kings,NaN,6.0,win,GJ Maxwell,494,4,494,494,71.0


In [3]:
run_sql_file('../sql/practice/02_merge_categories.sql')



ran ../sql/practice/02_merge_categories.sql


In [4]:
q("""
SELECT COUNT(DISTINCT bowler_type) 
FROM deliveries;
""")

,COUNT(DISTINCT bowler_type)
0,21


In [5]:
q("""
SELECT COUNT(DISTINCT bowler_type_clean)
FROM v_deliveries_clean;
""")

,COUNT(DISTINCT bowler_type_clean)
0,21


In [6]:
run_sql_file(r'..\sql\practice\03_venue_names.sql')

ran ..\sql\practice\03_venue_names.sql


In [7]:
q("""
SELECT COUNT(DISTINCT
    TRIM(
        SUBSTR(
            venue,
            1,
            CASE
                WHEN INSTR(venue, ',') > 0
                THEN INSTR(venue, ',') - 1
                ELSE LENGTH(venue)
            END
        )
    )
) AS after_rule
FROM matches;
""")

,after_rule
0,42


In [8]:
q("""
SELECT COUNT(DISTINCT venue_clean)
FROM v_matches_clean;
""")

,COUNT(DISTINCT venue_clean)
0,41


In [9]:
run_sql_file(r'..\sql\practice\04_dedupe_venues.sql')

ran ..\sql\practice\04_dedupe_venues.sql


In [10]:
q(""" 
SELECT COUNT(*) AS row_count 
FROM v_venues_clean; 
""")

DatabaseError: Execution failed on sql ' 
SELECT COUNT(*) AS row_count 
FROM v_venues_clean; 
': no such table: v_venues_clean

In [ ]:
q(""" 
SELECT COUNT(*) AS missing_city 
FROM v_venues_clean WHERE city IS NULL OR TRIM(city) = ''; 
""")

,missing_city
0,0


In [ ]:
q(""" 
SELECT COUNT(*) AS joined_rows 
FROM matches 
m JOIN v_venues_clean v ON v.venue = m.venue; 
""")

,joined_rows
0,1212


In [ ]:
con.executescript("""
DROP VIEW IF EXISTS v_matches_city_clean;
CREATE VIEW v_matches_city_clean AS
SELECT
    m.*,
    COALESCE(v.city, m.city, 'UNKNOWN') AS city_clean
FROM matches AS m
LEFT JOIN v_venues_clean AS v
    ON v.venue = m.venue;
""")
con.commit()
q("SELECT * FROM v_matches_city_clean LIMIT 5;")

ran ../sql/practice/05_city_fallback.sql


In [ ]:
run_sql_file('../sql/practice/06_season_year.sql')

ran ../sql/practice/06_season_year.sql


In [ ]:
q(""" SELECT    season,    season_year FROM v_matches_season_clean LIMIT 10; """)

,season,season_year
0,2008,2008
1,2008,2008
2,2008,2008
3,2008,2008
4,2008,2008
5,2008,2008
6,2008,2008
7,2008,2008
8,2008,2008
9,2008,2008


In [ ]:
run_sql_file('../sql/practice/07_win_definition.sql')

ran ../sql/practice/07_win_definition.sql


In [ ]:
q(""" SELECT COUNT(*) AS wins FROM matches WHERE result = 'win'; """)

,wins
0,1187


In [ ]:
run_sql_file('../sql/practice/01_nulls_and_blanks.sql')
run_sql_file('../sql/practice/02_merge_categories.sql')
run_sql_file('../sql/practice/03_venue_names.sql')
run_sql_file('../sql/practice/04_dedupe_venues.sql')
run_sql_file('../sql/practice/05_city_fallback.sql')
run_sql_file('../sql/practice/06_season_year.sql')
run_sql_file('../sql/practice/07_win_definition.sql')
run_sql_file('../sql/practice/08_matches_clean.sql')

ran ../sql/practice/01_nulls_and_blanks.sql


ran ../sql/practice/02_merge_categories.sql
ran ../sql/practice/03_venue_names.sql
ran ../sql/practice/04_dedupe_venues.sql
ran ../sql/practice/05_city_fallback.sql
ran ../sql/practice/06_season_year.sql
ran ../sql/practice/07_win_definition.sql
ran ../sql/practice/08_matches_clean.sql


In [ ]:
q("""
SELECT name, type
FROM sqlite_master
WHERE type IN ('table', 'view')
ORDER BY name;
""")

,name,type
0,deliveries,table
1,matches,table
2,matches_clean,table
3,players,table
4,teams,table
5,v_deliveries_clean,view
6,v_example,view
7,v_matches_city_clean,view
8,v_matches_clean,view
9,v_matches_season_clean,view
